# Exercise 7: Partitioning Strategies — Chromosome as a Natural Key
## BINFX410 — Chapter 10

---

## Learning Objectives
- Explain why chromosome is a near-ideal partition key for genomic variant data
- Implement three different partitioning strategies for the same dataset
- Measure the impact of partitioning on Athena bytes scanned and query cost
- Diagnose the small-files problem using S3 object listing
- Apply partition pruning rules to choose appropriate partition keys

**Estimated time:** 60 minutes  
**Dataset:** 1000 Genomes Project chr22 (`s3://1000genomes`) — variants from 2,504 individuals  
**AWS services:** S3, Athena, Glue Data Catalog

## Background: Why Partitioning Matters for Genomics

A full-genome variant dataset for 2,504 individuals contains ~80 million variants across 24 chromosomes. If stored in one big Parquet file:

- Query `WHERE chrom = 'chr22'`: scans all 80M rows, reads ~40 GB
- chr22 is ~1.7% of the genome → you're scanning 98.3% unnecessary data

With chromosome partitioning:
- Athena reads the `chrom=chr22/` prefix only
- Scans ~680 MB instead of ~40 GB
- **59x less data, 59x cheaper**

### Why Chromosome is an Ideal Partition Key

| Property | Ideal Value | Chromosome |
|---|---|---|
| Cardinality | 10–1000 values | 24 (chr1–22, chrX, chrY) ✓ |
| Query filter usage | Almost always filtered | Nearly all genomics queries filter by chr ✓ |
| Even data distribution | Balanced file sizes | Roughly proportional to chr length ✓ |
| Biological meaning | Partition = analysis unit | Yes — analyses are chr-scoped ✓ |

Compare to a bad partition key: sample_id (2,504 values × 24 chrs = 60,096 tiny files)

## Section 1: Setup

**What this does:** This installs the required Python packages. `pyarrow` provides the Parquet read/write engine, `matplotlib` is used to visualize the benchmark results in Section 7, and the others provide AWS SDK and DataFrame support.

```python
!pip install boto3 awswrangler pandas pyarrow matplotlib --quiet
```

**What this does:** This initializes all AWS SDK clients and sets the per-student S3 bucket and Athena results path. An unsigned S3 client is also created for accessing the public `1000genomes` bucket without credentials. No AWS resources are created yet.

```python
import boto3
import awswrangler as wr
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import io
import time
from botocore import UNSIGNED
from botocore.config import Config

STUDENT_ID = "sab032226"  # CHANGE THIS
AWS_REGION = "us-east-1"
BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

s3 = boto3.client('s3', region_name=AWS_REGION)
s3_unsigned = boto3.client('s3', region_name=AWS_REGION,
                            config=Config(signature_version=UNSIGNED))
session = boto3.Session(region_name=AWS_REGION)

print("Setup complete.")
```

## Section 2: Exploring 1000 Genomes Data

**What this does:** This uses the unsigned S3 client to list objects in the public `1000genomes` bucket under the `release/20130502/` prefix, specifically looking for chr22 VCF files. No data is downloaded; this is a metadata-only `ListObjectsV2` API call that returns file names and sizes, giving you a sense of the scale of the real dataset before generating a synthetic subset.

```python
# List 1000 Genomes VCF files
response = s3_unsigned.list_objects_v2(
    Bucket='1000genomes',
    Prefix='release/20130502/',
    Delimiter='/'
)

# Find chromosome-specific VCF files
all_files = s3_unsigned.list_objects_v2(
    Bucket='1000genomes',
    Prefix='release/20130502/ALL.chr22',
    MaxKeys=10
)

print("1000 Genomes chr22 files:")
for obj in all_files.get('Contents', []):
    size_mb = obj['Size'] / 1024 / 1024
    print(f"  {obj['Key'].split('/')[-1]}  ({size_mb:.0f} MB)")
```

**What this does:** This generates a realistic synthetic dataset of 10,000 chr22 variants entirely in local memory using NumPy and pandas — no S3 or Athena calls are made. The DataFrame includes variant position, type (SNP/INDEL), allele frequency, quality score, superpopulation, population code, and functional annotation, mirroring the structure of real 1000 Genomes VCF data. This local DataFrame is the source for all three partitioning strategy writes in Sections 3–5.

```python
# Build a representative chr22 variant dataset
# (In production: download the full 1.1 GB VCF and parse with pysam or cyvcf2)
# For this exercise, we generate a realistic synthetic chr22 variant set

import numpy as np
np.random.seed(42)

n_variants = 10000  # realistic subset for the exercise

# 1000 Genomes populations
populations = {
    'AFR': ['YRI', 'LWK', 'GWD', 'MSL', 'ESN'],
    'AMR': ['MXL', 'PUR', 'CLM', 'PEL'],
    'EAS': ['CHB', 'JPT', 'CHS', 'CDX', 'KHV'],
    'EUR': ['CEU', 'TSI', 'FIN', 'GBR', 'IBS'],
    'SAS': ['GIH', 'PJL', 'BEB', 'STU', 'ITU']
}

sample_pops = []
for superpop, subpops in populations.items():
    for subpop in subpops:
        sample_pops.extend([{'super': superpop, 'pop': subpop}] * 100)  # ~100 samples per pop

variant_types = ['SNP', 'INDEL']
chr22_length = 50_818_468

variants_df = pd.DataFrame({
    'chrom': 'chr22',
    'pos': np.random.randint(16000000, chr22_length, size=n_variants),
    'ref': np.random.choice(['A', 'T', 'G', 'C'], size=n_variants),
    'alt': np.random.choice(['A', 'T', 'G', 'C', 'ATG', 'ATGC'], size=n_variants),
    'variant_type': np.random.choice(['SNP', 'SNP', 'SNP', 'INDEL'], size=n_variants),
    'af': np.random.beta(0.5, 2, size=n_variants).round(4),  # minor allele frequency
    'qual': np.random.randint(20, 1000, size=n_variants).astype(float),
    'superpopulation': np.random.choice(list(populations.keys()), size=n_variants),
    'population': np.random.choice([p for pops in populations.values() for p in pops], size=n_variants),
    'annotation': np.random.choice(['intergenic', 'intronic', 'synonymous', 'missense', 'UTR'], size=n_variants)
})

# Sort by position (as real VCFs are)
variants_df = variants_df.sort_values('pos').reset_index(drop=True)

print(f"Generated {len(variants_df):,} synthetic chr22 variants")
print(f"\nVariant type distribution:")
print(variants_df['variant_type'].value_counts().to_string())
print(f"\nAllele frequency range: {variants_df['af'].min():.4f} – {variants_df['af'].max():.4f}")
```

## Section 3: Strategy 1 — No Partitioning

**What this does:** This combines the chr22 synthetic data with two additional chromosome subsets (chr1 and chrX) to create a multi-chromosome dataset, then writes it as a single monolithic Parquet file to `s3://{BUCKET}/silver/variants/strategy=no_partitioning/all_variants.parquet`. With no partitioning, Athena must scan the entire file for any query regardless of which chromosome is filtered — this represents the baseline (worst-case) strategy for comparison.

```python
# Write entire dataset as a single Parquet file — no partitioning
# Also add a few other chromosomes to make the exercise realistic

# Add small sets of chr1 and chrX data
chr1_variants = variants_df.copy()
chr1_variants['chrom'] = 'chr1'
chr1_variants['pos'] = np.random.randint(1000000, 248_956_422, size=n_variants)

chrx_variants = variants_df.copy()
chrx_variants['chrom'] = 'chrX'
chrx_variants['pos'] = np.random.randint(1000000, 154_259_566, size=n_variants)

all_variants = pd.concat([variants_df, chr1_variants, chrx_variants], ignore_index=True)
print(f"Total variants across all chromosomes: {len(all_variants):,}")
print(f"Chromosomes: {sorted(all_variants['chrom'].unique())}")

# Strategy 1: Single file, no partitioning
buffer = io.BytesIO()
all_variants.to_parquet(buffer, engine='pyarrow', compression='snappy', index=False)
buffer.seek(0)

s3.put_object(
    Bucket=BUCKET,
    Key='silver/variants/strategy=no_partitioning/all_variants.parquet',
    Body=buffer.read()
)

no_part_size = s3.head_object(
    Bucket=BUCKET,
    Key='silver/variants/strategy=no_partitioning/all_variants.parquet'
)['ContentLength'] / 1024 / 1024

print(f"\nStrategy 1 written: 1 file, {no_part_size:.2f} MB")
```

## Section 4: Strategy 2 — Partitioned by Chromosome

**What this does:** This writes the same multi-chromosome dataset to S3 using `awswrangler` with `partition_cols=['chrom']`, creating one Parquet file per chromosome under S3 prefixes like `chrom=chr1/`, `chrom=chr22/`, and `chrom=chrX/`. It also registers the table in the Glue Data Catalog as `binfx410_silver.variants_by_chr` so Athena can use partition pruning — queries with `WHERE chrom = 'chr22'` will skip all other partition directories entirely.

```python
# Write data partitioned by chromosome
wr.s3.to_parquet(
    df=all_variants,
    path=f"s3://{BUCKET}/silver/variants/strategy=chr_partitioned/",
    dataset=True,
    database='binfx410_silver',
    table='variants_by_chr',
    partition_cols=['chrom'],
    boto3_session=session,
    compression='snappy',
    mode='overwrite'
)

# List the partition files
chr_parts = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix='silver/variants/strategy=chr_partitioned/'
)

print("Strategy 2: Partitioned by chromosome")
for obj in chr_parts.get('Contents', []):
    if obj['Key'].endswith('.parquet') or obj['Key'].endswith('.snappy.parquet'):
        print(f"  {obj['Key'].replace(f'silver/variants/strategy=chr_partitioned/', '')}: "
              f"{obj['Size']/1024/1024:.2f} MB")
```

## Section 5: Strategy 3 — Over-Partitioned

**What this does:** This writes the dataset partitioned by three columns simultaneously — `chrom`, `superpopulation`, and `variant_type` — producing up to 30 S3 prefix combinations (3 chromosomes × 5 superpopulations × 2 variant types). This demonstrates the small-files problem: each resulting Parquet file is only a few hundred kilobytes, far below the 128 MB target. Athena must issue a separate S3 file-open request for each file, and the overhead of opening 30 tiny files can exceed the time saved by partition pruning for many query patterns.

```python
# Over-partition: chromosome + superpopulation + variant_type
# This creates 3 chrom × 5 superpops × 2 types = 30 partitions
# With real 1000G data this would be thousands of tiny files

wr.s3.to_parquet(
    df=all_variants,
    path=f"s3://{BUCKET}/silver/variants/strategy=over_partitioned/",
    dataset=True,
    database='binfx410_silver',
    table='variants_over_partitioned',
    partition_cols=['chrom', 'superpopulation', 'variant_type'],
    boto3_session=session,
    compression='snappy',
    mode='overwrite'
)

# Count files and sizes
over_parts = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix='silver/variants/strategy=over_partitioned/'
)

over_files = [obj for obj in over_parts.get('Contents', [])
              if not obj['Key'].endswith('/')]
over_sizes = [obj['Size'] for obj in over_files]

print("Strategy 3: Over-partitioned (chrom + superpopulation + variant_type)")
print(f"  Files: {len(over_files)}")
if over_sizes:
    print(f"  Avg size: {sum(over_sizes)/len(over_sizes)/1024:.1f} KB")
    print(f"  Min size: {min(over_sizes)/1024:.1f} KB")
    print(f"  Max size: {max(over_sizes)/1024:.1f} KB")
print()
print("The small-files problem: Athena must open and close each file separately.")
print("File open overhead dominates when files are < 128 MB each.")
```

## Section 6: Register All Tables and Run Benchmarks

**What this does:** This registers the no-partition dataset in the Glue Data Catalog as `binfx410_silver.variants_no_partition`, making it queryable by Athena alongside the two partitioned tables already registered. All three tables now exist in the same Glue database and reference the same logical data, enabling apples-to-apples benchmark comparisons in the next cell.

```python
# Register no-partition table in Glue
wr.s3.to_parquet(
    df=all_variants,
    path=f"s3://{BUCKET}/silver/variants/strategy=no_partitioning/",
    dataset=True,
    database='binfx410_silver',
    table='variants_no_partition',
    boto3_session=session,
    compression='snappy',
    mode='overwrite'
)
print("All three tables registered in Glue catalog.")
```

**What this does:** This defines a Python helper that submits an Athena query, waits for it to complete, and returns a dictionary containing the table name, megabytes scanned (from the Athena execution statistics), wall-clock time, and estimated cost at $5.00 per TB scanned. This function is called in the next cell for every combination of query pattern and partitioning strategy to produce the benchmark comparison table.

```python
# Benchmark function
def benchmark_query(sql, table_name):
    start = time.time()
    
    qid = wr.athena.start_query_execution(
        sql=sql,
        database='binfx410_silver',
        s3_output=ATHENA_RESULTS,
        boto3_session=session
    )
    
    wr.athena.wait_query(query_execution_id=qid, boto3_session=session)
    elapsed = time.time() - start
    
    stats = boto3.client('athena', region_name=AWS_REGION).get_query_execution(
        QueryExecutionId=qid
    )['QueryExecution']['Statistics']
    
    return {
        'table': table_name,
        'bytes_scanned_mb': stats.get('DataScannedInBytes', 0) / 1024 / 1024,
        'wall_time_sec': round(elapsed, 2),
        'cost_usd': (stats.get('DataScannedInBytes', 0) / 1e12) * 5.0
    }

print("Benchmark helper defined.")
```

**What this does:** This runs three Athena queries — a chromosome filter (`WHERE chrom = 'chr22'`), a full-scan aggregation (`GROUP BY chrom`), and a two-column filter (`WHERE chrom = 'chr22' AND superpopulation = 'AFR'`) — against all three table variants and records the bytes scanned for each combination. For the chromosome-partitioned table, Athena's partition pruning skips irrelevant S3 prefixes, resulting in dramatically lower bytes scanned for the chromosome-filter query compared to the no-partition table.

```python
# Query 1: Chromosome-specific — this is where partitioning wins big
queries = {
    'q1_chr_filter': "SELECT COUNT(*) FROM {table} WHERE chrom = 'chr22'",
    'q2_full_scan':  "SELECT chrom, COUNT(*) as n FROM {table} GROUP BY chrom",
    'q3_multifilter': "SELECT COUNT(*) FROM {table} WHERE chrom = 'chr22' AND superpopulation = 'AFR'",
}

tables = {
    'No Partition': 'variants_no_partition',
    'Chr Partitioned': 'variants_by_chr',
    'Over-Partitioned': 'variants_over_partitioned'
}

all_results = []

for q_name, q_template in queries.items():
    print(f"\nRunning: {q_name}")
    for table_label, table_name in tables.items():
        sql = q_template.format(table=table_name)
        result = benchmark_query(sql, table_label)
        result['query'] = q_name
        all_results.append(result)
        print(f"  {table_label}: {result['bytes_scanned_mb']:.2f} MB scanned")

results_df = pd.DataFrame(all_results)
print("\n=== Complete Results ===")
results_df
```

## Section 7: Visualizing the Results

**What this does:** This pivots the benchmark results DataFrame and renders a grouped bar chart using matplotlib, with one bar group per query pattern and one bar per partitioning strategy. The chart makes the cost/performance tradeoffs immediately visible: the chromosome-partitioned strategy dramatically reduces bytes scanned for chromosome-filtered queries (Q1, Q3) but offers no advantage for full-table scans (Q2), while the over-partitioned strategy often performs similarly to or worse than the simple chromosome partition due to small-file overhead.

```python
pivot = results_df.pivot(index='query', columns='table', values='bytes_scanned_mb')
pivot.index = ['Q1: WHERE chrom=chr22', 'Q2: GROUP BY chrom', 'Q3: chrom + superpop']

ax = pivot.plot(
    kind='bar',
    figsize=(9, 4),
    color=['#4CAF50', '#F44336', '#FF9800'],
    edgecolor='white',
    width=0.6
)
ax.set_title('MB Scanned by Partitioning Strategy', fontweight='bold')
ax.set_ylabel('MB Scanned')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Strategy', bbox_to_anchor=(1.01, 1), loc='upper left')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)

plt.tight_layout()
plt.show()
```

## Section 8: Diagnosing the Small Files Problem

**What this does:** This uses the S3 `ListObjectsV2` paginator to enumerate every Parquet file under each of the three strategy prefixes, then computes file count, total size, average size, minimum size, and maximum size for each strategy. The output makes the small-files problem concrete: the over-partitioned strategy produces many files averaging well below 128 MB, while the no-partition and chromosome-partitioned strategies produce fewer, larger files that Athena can read more efficiently.

```python
# Analyze file counts and sizes for each strategy
strategies = {
    'No Partition': 'silver/variants/strategy=no_partitioning/',
    'Chr Partitioned': 'silver/variants/strategy=chr_partitioned/',
    'Over-Partitioned': 'silver/variants/strategy=over_partitioned/'
}

file_stats = []
for strategy, prefix in strategies.items():
    paginator = s3.get_paginator('list_objects_v2')
    objects = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get('Contents', []):
            if not obj['Key'].endswith('/'):
                objects.append(obj['Size'])
    
    if objects:
        file_stats.append({
            'strategy': strategy,
            'file_count': len(objects),
            'total_mb': sum(objects) / 1024 / 1024,
            'avg_kb': sum(objects) / len(objects) / 1024,
            'min_kb': min(objects) / 1024,
            'max_kb': max(objects) / 1024
        })

stats_df = pd.DataFrame(file_stats).set_index('strategy')
print("File statistics per partitioning strategy:")
print(stats_df.round(2).to_string())
print()
print("Rule of thumb: target file size > 128 MB for optimal Athena performance.")
print("Files < 1 MB indicate severe small-files problem.")
```

## Section 9: Best Practices Summary

**What this does:** This constructs a local pandas DataFrame summarizing six partitioning best practices for genomics data lakes, including target file sizes, recommended partition cardinality, and the specific recommendation to always partition genomic variant data by chromosome. No AWS resources are created; this cell runs entirely locally and displays as a formatted reference table.

```python
best_practices = pd.DataFrame([
    {'Rule': 'Target file size', 'Recommendation': '128 MB – 1 GB per Parquet file',
     'Why': 'Below 128 MB, S3 file open overhead dominates query time'},
    {'Rule': 'Partition cardinality', 'Recommendation': 'Fewer than 10,000 partitions total',
     'Why': 'Each partition = S3 LIST call; too many partitions slow catalog operations'},
    {'Rule': 'Partition key = filter key', 'Recommendation': 'Partition on columns your queries filter by',
     'Why': 'If you never filter by superpopulation, partitioning by it helps nothing'},
    {'Rule': 'For genomics: always partition by chromosome', 'Recommendation': 'chrom=chr1/, chrom=chr2/, ...',
     'Why': 'Nearly every genomics analysis is chromosome-scoped; 24 partitions is ideal'},
    {'Rule': 'Secondary partition key', 'Recommendation': 'Add tissue_type, cancer_type, or study as secondary',
     'Why': 'Only if your queries consistently filter by both chrom AND the secondary key'},
    {'Rule': 'Avoid partitioning by sample_id', 'Recommendation': 'Never partition by sample_id alone',
     'Why': '2,504 samples × 24 chrs = 60,096 files; pathological small-files case'},
])

best_practices.set_index('Rule', inplace=True)
print("Partitioning Best Practices for Genomics Data:")
best_practices
```

## Exercise: YOUR CODE HERE

**What this does:** When implemented, this would write the dataset to S3 partitioned by both `chrom` and `variant_type` (6 partitions: 3 chromosomes × 2 variant types), register it as a fourth Glue table, and run the same three benchmark queries against it. The results would show whether the additional `variant_type` partition dimension improves Q3 (which filters on both chrom and superpopulation) or merely increases file overhead without query benefit.

```python
# TASK 1: Create a fourth partitioning strategy: chromosome + variant_type
# Write the data, register as a Glue table, and run all 3 benchmark queries against it
# Is this better or worse than 'chr_partitioned' for each query type?

# YOUR CODE HERE
```

**What this does:** When implemented, this would scale the benchmark results (measured on 30,000 variants) up to the full 40 GB 1000 Genomes dataset and compute the projected Athena cost per query for the no-partition vs. chromosome-partitioned strategies. At $5.00 per TB scanned, the difference between scanning 40 GB (no partition) and ~680 MB (chr22 partition) for Q1 translates to a concrete dollar difference that accumulates quickly across thousands of daily queries.

```python
# TASK 2: Compute the projected cost difference at scale
# Assume the full 1000 Genomes dataset is 40 GB
# Scale your benchmark results (measured on 30K variants) to estimate:
#   - Cost per query with no partitioning
#   - Cost per query with chr partitioning
# for Q1 (chromosome filter) and Q2 (full scan)

# YOUR CODE HERE
full_dataset_gb = 40.0
athena_per_tb = 5.00
# ...
```

**What this does:** When implemented, this would read all Parquet files from the `variants_over_partitioned` prefix for `chrom=chr22` only using `awswrangler`, concatenate them into a single DataFrame, and write that DataFrame back as one consolidated Parquet file — effectively compacting the over-partitioned chr22 data into a single right-sized file. Measuring the before and after S3 object sizes demonstrates the storage and performance benefit of periodic compaction (equivalent to running `OPTIMIZE` on an Iceberg table).

```python
# TASK 3: For the over-partitioned dataset, implement a compaction step
# Read all files from variants_over_partitioned for chrom=chr22 only
# Write them back as a single Parquet file (removing the over-partitioned structure)
# Measure the before and after file sizes

# YOUR CODE HERE
```

## Reflection Questions

*(Double-click to edit)*

1. **Query 2 (full scan GROUP BY chrom) scanned the same amount of data regardless of partitioning.** Why? When would partitioning NOT help, and how do you design queries that benefit from it?

2. **Real 1000 Genomes data has 2,504 samples.** If you stored individual-level genotypes (not just variant positions) and partitioned by sample, you'd have 2,504 files per chromosome. What's the total file count for the full genome? Is this acceptable? What would you do instead?

3. **Partition projection** is an Athena feature that allows you to define the partition values computationally rather than registering each one in Glue. For a dataset partitioned by chromosome, this would mean Athena knows the valid values are chr1-chr22, chrX, chrY without having to query the catalog. When would partition projection be particularly valuable?

4. **You're designing a lakehouse for a clinical genetics lab** that runs two types of queries daily:
   - 90%: `WHERE sample_id = 'P-123' AND chrom = 'chr17'` (single patient, specific gene region)
   - 10%: `WHERE gene_symbol = 'BRCA1'` (all samples, specific gene)
   
   What partitioning strategy would you choose? Is there a single optimal choice for both query patterns?

5. **The Iceberg `OPTIMIZE` command compacts small files** after they accumulate from many small writes. Under what operational scenario for a genomics lab would small files accumulate organically over time, and how frequently would you run OPTIMIZE?

*(Write answers here)*

1. 

2. 

3. 

4. 

5. 